# Oracle Root-Cause Diagnostic

Notebook này không cố sửa model ngay.

Nó dùng `Stockfish fixed-node oracle` trên một subset chẩn đoán để trả lời trực tiếp:

- train label hiện tại lệch khỏi oracle mạnh đến mức nào
- teacher có đang sai thật, hay đang “sai so với label nhưng gần oracle hơn”
- volatility có liên quan tới error của teacher hay không
- trên stable subset, lỗi calibration còn lại lớn đến đâu
- scale `tanh(cp / c)` nào khớp teacher hơn trên oracle cố định

In [1]:
from dataclasses import asdict
from pathlib import Path
import json
import sys

import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path(r"C:\Users\USER\Desktop\chess_engine")
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "oracle_root_cause_diagnostic"
RUN_DIR = Path(r"C:\Users\USER\Downloads\dgrn_5m_v3_stage2_polish_run1")
DATA_ROOT = PROJECT_ROOT / "data" / "process"

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

import oracle_diagnostic_helpers as lab

torch.set_float32_matmul_precision("high")
lab.set_global_seed(123)
paths = lab.build_default_paths(run_dir=RUN_DIR, data_root=DATA_ROOT, experiment_dir=EXPERIMENT_DIR)
lab.export_paths_json(paths, paths["output_dir"] / "paths.json")

if "envs\\chess_engine" not in sys.executable.lower():
    raise RuntimeError(
        f"Notebook is running under the wrong interpreter: {sys.executable}. "
        "Select the 'chess_engine' Jupyter kernel."
    )

DEVICE = lab.choose_device(prefer_cuda=True)
if DEVICE.type != "cuda":
    raise RuntimeError("CUDA is required for this notebook.")

CHECKPOINT = paths["run_dir"] / "ckpt_best.pt"
assert CHECKPOINT.exists(), f"Missing checkpoint: {CHECKPOINT}"

CFG = lab.OracleDiagnosticConfig(
    split="test",
    sample_abs_y_edges=(0.0, 0.05, 0.20, 0.50, 0.70, 1.00),
    sample_per_band=48,
    err_quantiles=(1.0 / 3.0, 2.0 / 3.0),
    oracle_scales=(400.0, 600.0, 800.0, 1200.0),
    stockfish_path=r"D:\stockfish-windows-x86-64-avx2\stockfish\stockfish-windows-x86-64-avx2.exe",
    stockfish_threads=1,
    stockfish_hash_mb=32,
    stockfish_node_budgets=(8_000, 32_000, 128_000),
    stockfish_command_pause_ms=50,
    stockfish_timeout_sec=20.0,
    prediction_batch_size=2048,
    sample_seed=123,
    benchmark_train_batch_size=640,
    decode_validation_samples=64,
    subset_num_shards=None,
)
cfg_validation = lab.validate_diagnostic_config(CFG)
lab.save_json(asdict(CFG), paths["output_dir"] / "runtime_config.json")
lab.save_json(cfg_validation, paths["reports_dir"] / "config_validation.json")

print("python:", sys.executable)
print("device:", DEVICE)
print("gpu_name:", torch.cuda.get_device_name(0))
display(pd.DataFrame({"path_key": list(paths.keys()), "path_value": [str(v) for v in paths.values()]}))
display(pd.DataFrame([cfg_validation]))

python: c:\Users\USER\anaconda3\envs\chess_engine\python.exe
device: cuda
gpu_name: NVIDIA GeForce RTX 2050


,path_key,path_value
0,project_root,C:\Users\USER\Desktop\chess_engine
1,run_dir,C:\Users\USER\Downloads\dgrn_5m_v3_stage2_poli...
2,data_root,C:\Users\USER\Desktop\chess_engine\data\process
3,experiment_dir,C:\Users\USER\Desktop\chess_engine\experiments...
4,output_dir,C:\Users\USER\Desktop\chess_engine\experiments...
5,plots_dir,C:\Users\USER\Desktop\chess_engine\experiments...
6,reports_dir,C:\Users\USER\Desktop\chess_engine\experiments...
7,cache_dir,C:\Users\USER\Desktop\chess_engine\experiments...
8,checkpoints_dir,C:\Users\USER\Desktop\chess_engine\experiments...
9,split_pred_cache_dir,C:\Users\USER\Desktop\chess_engine\experiments...


,is_valid,issues,stockfish_validation
0,True,[],"{'is_valid': True, 'near_zero_thr': 0.2, 'cali..."


## Runtime Self-Check

Cell này kiểm tra:
- GPU forward benchmark
- Stockfish determinism
- decode -> FEN compatibility

In [2]:
benchmark = lab.benchmark_single_train_step(
    init_ckpt_path=CHECKPOINT,
    data_root=paths["data_root"],
    device=DEVICE,
    cfg=CFG,
)
lab.save_json(benchmark, paths["reports_dir"] / "runtime_benchmark.json")

sf_cfg = lab.build_stockfish_cfg(CFG)
stockfish_benchmark = lab.sw_lab.benchmark_stockfish_proxy(sf_cfg)
stockfish_validation = lab.sw_lab.validate_stockfish_proxy(sf_cfg)
stockfish_decode = lab.sw_lab.validate_stockfish_compatible_decoding(
    data_root=paths["data_root"],
    split=CFG.split,
    sample_count=CFG.decode_validation_samples,
    num_shards=1,
)
stockfish_dataset_validation = lab.sw_lab.validate_stockfish_proxy_on_dataset_sample(
    data_root=paths["data_root"],
    split=CFG.split,
    cfg=sf_cfg,
    sample_index=0,
    num_shards=1,
)

lab.save_json(stockfish_benchmark, paths["reports_dir"] / "stockfish_proxy_benchmark.json")
lab.save_json(stockfish_validation, paths["reports_dir"] / "stockfish_proxy_validation.json")
lab.save_json(stockfish_decode, paths["reports_dir"] / "stockfish_decode_validation.json")
lab.save_json(stockfish_dataset_validation, paths["reports_dir"] / "stockfish_proxy_dataset_validation.json")

split_summary = pd.DataFrame(
    [
        lab.sw_lab.summarize_split_layout(paths["data_root"], "train"),
        lab.sw_lab.summarize_split_layout(paths["data_root"], "val"),
        lab.sw_lab.summarize_split_layout(paths["data_root"], "test"),
    ]
)
lab.save_dataframe(split_summary, paths["reports_dir"] / "split_summary.csv")
assert stockfish_validation["bestmove_match"], stockfish_validation
assert stockfish_validation["target_match"], stockfish_validation
if stockfish_decode["checked_with_python_chess"]:
    assert stockfish_decode["invalid_fens"] == 0, stockfish_decode
assert stockfish_dataset_validation["bestmove_match"], stockfish_dataset_validation
assert stockfish_dataset_validation["target_match"], stockfish_dataset_validation
display(pd.DataFrame([benchmark]))
display(pd.DataFrame(stockfish_benchmark["per_query"]))
display(pd.DataFrame([stockfish_validation]))
display(pd.DataFrame([stockfish_dataset_validation]))
display(pd.DataFrame([stockfish_decode]))
display(split_summary)

,batch_size,step_time_sec,peak_mem_gb,steps_per_epoch,train_total_samples,train_num_shards,epoch_hours_estimate
0,640,4.08275,3.600463,79,50000,1,0.089594


,node_budget,elapsed_sec,target_value,bestmove
0,8000,0.461087,0.088104,e2e4
1,32000,0.458422,0.061589,e2e4
2,128000,0.589093,0.056606,e2e4


,probe_fen,node_budgets,first_bestmoves,second_bestmoves,first_targets,second_targets,bestmove_match,target_match
0,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,"[8000, 32000, 128000]","[e2e4, e2e4, e2e4]","[e2e4, e2e4, e2e4]","[0.08810429968944254, 0.061588617176587195, 0....","[0.08810429968944254, 0.061588617176587195, 0....",True,True


,probe_fen,node_budgets,first_bestmoves,second_bestmoves,first_targets,second_targets,bestmove_match,target_match,split,sample_index
0,r4rk1/b2P2p1/p2Rqn1p/2p2N2/4P3/4B2P/2B1QPP1/6K...,"[8000, 32000, 128000]","[e6e5, e6e5, e6e5]","[e6e5, e6e5, e6e5]","[-0.7130362273932255, -0.6953979670996447, -0....","[-0.7130362273932255, -0.6953979670996447, -0....",True,True,test,0


,sample_count,invalid_fens,checked_with_python_chess,examples
0,64,0,True,[]


,split,num_shards,samples
0,train,80,4000000
1,val,10,500000
2,test,10,500000


## Build Diagnostic Subset

Trình tự:
1. chạy teacher hiện tại trên toàn bộ split chẩn đoán
2. stratify theo `|y|` và `teacher_abs_err`
3. lấy subset cân bằng để hỏi Stockfish oracle

In [3]:
pred_cache = lab.precompute_split_prediction_cache(
    init_ckpt_path=CHECKPOINT,
    data_root=paths["data_root"],
    split=CFG.split,
    output_dir=paths["output_dir"],
    device=DEVICE,
    batch_size=CFG.prediction_batch_size,
    num_shards=CFG.subset_num_shards,
)

subset = lab.build_stratified_subset(
    data_root=paths["data_root"],
    pred_cache_dir=paths["split_pred_cache_dir"],
    split=CFG.split,
    cfg=CFG,
    num_shards=CFG.subset_num_shards,
)
lab.save_dataframe(subset["count_table"], paths["reports_dir"] / "subset_candidate_count_table.csv")
lab.save_dataframe(subset["quota_table"], paths["reports_dir"] / "subset_quota_table.csv")
lab.save_dataframe(subset["sampled_summary"], paths["reports_dir"] / "subset_sampled_summary.csv")
display(subset["sampled_summary"])
display(subset["quota_table"].head(15))

[test_pred_cache_00000] offset=0 / 50000 elapsed=10.3s
[test-pred-cache] shards=1/10 elapsed=241.0s
[test_pred_cache_00001] offset=0 / 50000 elapsed=9.9s
[test_pred_cache_00002] offset=0 / 50000 elapsed=10.5s
[test_pred_cache_00003] offset=0 / 50000 elapsed=9.8s
[test_pred_cache_00004] offset=0 / 50000 elapsed=10.0s
[test-pred-cache] shards=5/10 elapsed=1213.4s
[test_pred_cache_00005] offset=0 / 50000 elapsed=9.9s
[test_pred_cache_00006] offset=0 / 50000 elapsed=9.8s
[test_pred_cache_00007] offset=0 / 50000 elapsed=9.8s
[test_pred_cache_00008] offset=0 / 50000 elapsed=9.8s
[test-pred-cache] shards=9/10 elapsed=2166.6s
[test_pred_cache_00009] offset=0 / 50000 elapsed=9.9s


,band_idx,band_label,selected
0,0,"[0.000,0.050]",48
1,1,"[0.050,0.200]",48
2,2,"[0.200,0.500]",48
3,3,"[0.500,0.700]",48
4,4,"[0.700,1.000]",48


,band_idx,band_label,err_bin_id,err_left,err_right,count,quota
0,0,"[0.000,0.050]",0,0.000000,0.050110,52342,16
1,0,"[0.000,0.050]",1,0.050110,0.135376,52351,16
2,0,"[0.000,0.050]",2,0.135376,1.002132,52349,16
3,1,"[0.050,0.200]",0,0.000000,0.050904,39319,16
4,1,"[0.050,0.200]",1,0.050904,0.124325,39319,16
5,1,"[0.050,0.200]",2,0.124325,1.049586,39320,16
6,2,"[0.200,0.500]",0,0.000000,0.097300,30333,16
7,2,"[0.200,0.500]",1,0.097300,0.215542,30333,16
8,2,"[0.200,0.500]",2,0.215542,1.432213,30334,16
9,3,"[0.500,0.700]",0,0.000000,0.135761,21333,16


## Run Oracle And Analyze

Cell này là phần quan trọng nhất.

Quy ước:
- `oracle reference` = kết quả ở node budget lớn nhất
- các node budget nhỏ hơn chỉ dùng để đo stability / disagreement curve

Nó tạo ra:
- `oracle_subset_rows.csv`
- `oracle_band_summary.csv`
- `oracle_budget_alignment.csv`
- `oracle_stability_summary.csv`
- `oracle_scale_sweep.csv`
- `oracle_root_cause_summary.json`

In [4]:
oracle_rows = lab.run_stockfish_oracle_on_subset(
    subset=subset["samples"],
    cfg=CFG,
    output_dir=paths["output_dir"],
)
analysis = lab.run_diagnostic_analysis(
    df=oracle_rows,
    cfg=CFG,
    output_dir=paths["output_dir"],
)

lab.save_dataframe(oracle_rows.head(120), paths["reports_dir"] / "oracle_subset_rows_preview.csv")
display(pd.DataFrame([analysis["summary"]]))
display(analysis["band_summary"])
display(analysis["budget_summary"])
display(analysis["stability_summary"])
display(analysis["scale_sweep"])
display(analysis["stable_bucket"].head(12))

[oracle-subset] processed=1/240
[oracle-subset] processed=13/240
[oracle-subset] processed=25/240
[oracle-subset] processed=37/240
[oracle-subset] processed=49/240
[oracle-subset] processed=61/240
[oracle-subset] processed=73/240
[oracle-subset] processed=85/240
[oracle-subset] processed=97/240
[oracle-subset] processed=109/240
[oracle-subset] processed=121/240
[oracle-subset] processed=133/240
[oracle-subset] processed=145/240
[oracle-subset] processed=157/240
[oracle-subset] processed=169/240
[oracle-subset] processed=181/240
[oracle-subset] processed=193/240
[oracle-subset] processed=205/240
[oracle-subset] processed=217/240
[oracle-subset] processed=229/240


,n_total,teacher_closer_to_oracle_rate_600_overall,teacher_closer_to_oracle_rate_600_near_zero,train_vs_oracle_mae_600_near_zero,teacher_vs_oracle_mae_600_near_zero,stable_near_teacher_vs_oracle_mae_600,unstable_near_teacher_vs_oracle_mae_600,stable_near_train_vs_oracle_mae_600,unstable_near_train_vs_oracle_mae_600,stable_0.7_slope_600,corr_teacher_oracle_vs_label_oracle_600,corr_instability_vs_teacher_oracle_600,corr_instability_vs_train_oracle_600,best_scale_by_group
0,240,0.216667,0.141414,0.036008,0.119898,0.116427,0.126469,0.047909,0.02993,0.575106,0.093107,-0.008725,-0.026819,"{'overall': {'scale': 800.0, 'mae': 0.16429833..."


,band_label,n,teacher_vs_train_mae,teacher_vs_oracle_mae_600,train_vs_oracle_mae_600,teacher_closer_to_oracle_rate_600,teacher_oracle_sign_match_rate,train_oracle_sign_match_rate,oracle_false_0.1_0.3,oracle_false_0.2_0.4,mean_band_instability_score
0,"[0.000,0.050]",48,0.121556,0.124146,0.028411,0.145833,0.562500,0.708333,0.065217,0.041667,0.500
1,"[0.050,0.200]",48,0.113286,0.118240,0.030825,0.083333,0.770833,1.000000,0.000000,0.046512,0.500
2,"[0.200,0.500]",48,0.203469,0.185025,0.058176,0.187500,0.854167,1.000000,NaN,0.000000,0.375
3,"[0.500,0.700]",48,0.278008,0.252348,0.071750,0.229167,0.875000,1.000000,NaN,NaN,0.375
4,"[0.700,1.000]",48,0.270325,0.218644,0.134458,0.437500,0.979167,1.000000,NaN,NaN,0.375


,band_label,node_budget,train_vs_oracle_budget_mae
0,"[0.000,0.050]",8000,0.041171
1,"[0.000,0.050]",32000,0.036415
2,"[0.000,0.050]",128000,0.028411
3,"[0.050,0.200]",8000,0.040004
4,"[0.050,0.200]",32000,0.031499
5,"[0.050,0.200]",128000,0.030825
6,"[0.200,0.500]",8000,0.081660
7,"[0.200,0.500]",32000,0.065915
8,"[0.200,0.500]",128000,0.058176
9,"[0.500,0.700]",8000,0.117968


,band_label,stability_group,n,teacher_vs_oracle_mae_600,train_vs_oracle_mae_600,teacher_closer_to_oracle_rate_600,teacher_oracle_sign_match_rate,train_oracle_sign_match_rate,oracle_false_0.1_0.3,oracle_false_0.2_0.4,mean_band_instability_score
0,"[0.000,0.050]",unstable,16,0.139581,0.029227,0.1875,0.3750,0.7500,0.062500,0.0625,0.747340
1,"[0.000,0.050]",stable,16,0.107023,0.027017,0.1250,0.4375,0.6875,0.000000,0.0000,0.250000
2,"[0.050,0.200]",stable,16,0.098153,0.030262,0.1875,0.6875,1.0000,0.000000,0.0625,0.253989
3,"[0.050,0.200]",middle,16,0.140949,0.038040,0.0625,0.8125,1.0000,0.000000,0.0000,0.524269
4,"[0.200,0.500]",middle,16,0.111892,0.034661,0.2500,1.0000,1.0000,NaN,NaN,0.366356
5,"[0.200,0.500]",stable,16,0.200566,0.086563,0.1875,0.7500,1.0000,NaN,0.0000,0.178524
6,"[0.500,0.700]",middle,16,0.255863,0.073497,0.3125,0.8125,1.0000,NaN,NaN,0.380652
7,"[0.500,0.700]",stable,16,0.291096,0.050089,0.1250,0.8750,1.0000,NaN,NaN,0.183178
8,"[0.700,1.000]",middle,16,0.191182,0.156314,0.5000,1.0000,1.0000,NaN,NaN,0.359043
9,"[0.700,1.000]",stable,16,0.150416,0.066523,0.4375,1.0000,1.0000,NaN,NaN,0.191157


,group,scale,n,mse,mae
0,overall,400.0,240,0.108245,0.246006
1,stable,400.0,80,0.104067,0.250812
2,stable_0.2,400.0,36,0.033444,0.149265
3,stable_0.7,400.0,70,0.112632,0.263143
4,overall,600.0,240,0.063490,0.179680
5,stable,600.0,80,0.055029,0.169451
6,stable_0.2,600.0,36,0.023351,0.116427
7,stable_0.7,600.0,70,0.059268,0.176814
8,overall,800.0,240,0.050855,0.164298
9,stable,800.0,80,0.039733,0.145880


,bucket,left,right,center,count,mse,mae,mean_y,mean_p,bias,abs_cal_gap
0,0,-1.0,-0.9,-0.95,0,NaN,NaN,NaN,NaN,NaN,NaN
1,1,-0.9,-0.8,-0.85,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2,-0.8,-0.7,-0.75,3,0.016370,0.114562,-0.708034,-0.721354,-0.013320,0.013320
3,3,-0.7,-0.6,-0.65,8,0.048320,0.182173,-0.661357,-0.494049,0.167308,0.167308
4,4,-0.6,-0.5,-0.55,3,0.046217,0.185641,-0.562566,-0.402018,0.160548,0.160548
5,5,-0.5,-0.4,-0.45,2,0.014684,0.104394,-0.440454,-0.336060,0.104394,0.104394
6,6,-0.4,-0.3,-0.35,4,0.066600,0.239303,-0.355907,-0.116604,0.239303,0.239303
7,7,-0.3,-0.2,-0.25,1,0.000482,0.021964,-0.289787,-0.267822,0.021964,0.021964
8,8,-0.2,-0.1,-0.15,5,0.032260,0.167797,-0.158773,0.009024,0.167797,0.167797
9,9,-0.1,0.0,-0.05,9,0.013037,0.098119,-0.042897,-0.021737,0.021161,0.021161
